# 🛰️ TRISAT (Notebook 02): Complete EUSPA Galileo OSNMA v1.1, E6 HAS-UA (PPP) & OS SIS v2.2 Suite

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jreyes13/TRISAT/blob/main/notebooks/02_euspa_osnma_has_complete_suite.ipynb)

This interactive notebook documents and executes the complete **TRISAT** and **TRISAT-SDR** architecture compliant with official **EUSPA (European Union Agency for the Space Programme)** reference specifications:

1. **Synchronized Dual-HackRF Multi-Band SDR Architecture:** PRN code generation and `CBOC(6,1,1/11)` modulation on **E1-B/C** ($1575.42\text{ MHz}$) + `BPSK(5)` on **E6-B/C HAS** ($1278.75\text{ MHz}$) + **E5a/E5b AltBOC**.
2. **Galileo OSNMA v1.1 Cryptographic Engine (Official Annex A Test Vectors):**
   - **Merkle Tree (`DSM-PKR`)** root and `P_DP` padding verification.
   - **NIST P-256 (`secp256r1`)** compressed public key decompression and **ECDSA-SHA256** signature verification over `DSM-KROOT`.
   - **TESLA One-Way Key Chain ($K_{14} \to \dots \to K_4 \to K_0$)** with salt $\alpha = \text{0x610BDF26D77B}$.
   - Verification of `MACSEQ = 0xB08` and 40-bit truncated `HMAC-SHA-256` tags (`ADKD=0, 4, 12`).
3. **Galileo High Accuracy Service Reference User Algorithm (`EUSPA HAS-UA v2.0`):**
   - **Reed-Solomon `RS(255, 32, d=224)`** decoding over $\text{GF}(2^8)$ ($p(x)=x^8+x^4+x^3+x^2+1$, `0x11D`).
   - **NTW $\to$ ECEF** orbital coordinate transformation (Eq. 19), **Pivot Satellite selection** (Eq. 80), **Single-Difference PPP (`PPPSD`)** EKF, and **DIA Quality Control (Global LOM F-test + Baarda $w$-test)**.
   - **12-Coefficient `NTCM-G` Ionospheric Model** for single-frequency users.
4. **Galileo OS SIS ICD v2.2 Advanced Services:**
   - `Word Type 16` (**RedCED** for fast TTFF), `Word Types 17..20` (**Reed-Solomon `RS(118,58)` Outer FEC2**), `Word Type 22` (**ARAIM ISM**), `Word Type 44` (**Timing Service Message - TSM**), and the 122-bit **`EWSS-CAMF`** emergency alert ellipse geo-fence.

## 0. Environment Setup (Self-Contained for Google Colab & Local Execution)

The cell below automatically clones the `TRISAT` repository when running inside **Google Colab**, or adds the local repository root to `sys.path`.

In [ ]:
import os
import sys
import math
import hashlib
import hmac
import numpy as np
import matplotlib.pyplot as plt

if not os.path.exists('trisat_sdr') and not os.path.exists('../trisat_sdr'):
    print('🌐 Google Colab environment detected. Cloning TRISAT repository...')
    !git clone --depth 1 https://github.com/jreyes13/TRISAT.git /tmp/TRISAT_repo 2>/dev/null || true
    if os.path.exists('/tmp/TRISAT_repo'):
        sys.path.insert(0, '/tmp/TRISAT_repo')
else:
    repo_root = os.path.abspath('..') if os.path.exists('../trisat_sdr') else os.path.abspath('.')
    if repo_root not in sys.path:
        sys.path.insert(0, repo_root)

plt.style.use('seaborn-v0_8-darkgrid' if 'seaborn-v0_8-darkgrid' in plt.style.available else 'default')
print('✅ TRISAT environment ready.')

---
## 1. Dual-HackRF RF Architecture & `CBOC(6,1,1/11)` vs `BPSK(5)` Modulation

In **TRISAT-SDR**, two SDR front-ends (**HackRF One #0 Master** and **HackRF One #1 Slave** phase-locked via the $10\text{ MHz}$ `CLKOUT -> CLKIN` reference clock) simultaneously capture:

| SDR Front-End | GNSS Band | Center Frequency | Modulation | Chip Rate | EUSPA Service |
| :--- | :--- | :---: | :---: | :---: | :--- |
| **HackRF #0 (Master)** | **Galileo E1-B/C + GPS L1** | $1575.42\text{ MHz}$ | `CBOC(6,1,1/11)` | $1.023\text{ Mcps}$ | **I/NAV v2.2 + OSNMA (40 bits)** |
| **HackRF #1 (Slave)** | **Galileo E6-B/C** | $1278.75\text{ MHz}$ | `BPSK(5)` | $5.115\text{ Mcps}$ | **C/NAV HAS (`RS(255,32)` PPP)** |
| **HackRF #1 (Alt.)** | **Galileo E5a / E5b / AltBOC** | $1176.45 / 1191.795\text{ MHz}$ | `AltBOC(15,10)` | $10.23\text{ Mcps}$ | **F/NAV + I/NAV Iono-Free** |

The **Galileo E1-B** signal uses in-phase Composite Binary Offset Carrier $\text{CBOC}(6, 1, 1/11)$ modulation:
$$s_{\text{E1-B}}(t) = \sqrt{\frac{10}{11}}\,\text{sc}_{\text{BOC}(1,1)}(t) - \sqrt{\frac{1}{11}}\,\text{sc}_{\text{BOC}(6,1)}(t)$$

In [ ]:
# Visualize Galileo E1 CBOC(6,1,1/11) subcarrier waveform and 5-Tap Correlator Autocorrelation Peak
t_chip = np.linspace(0, 2, 1000)
boc11 = np.sign(np.sin(2 * np.pi * 1.0 * t_chip))
boc61 = np.sign(np.sin(2 * np.pi * 6.0 * t_chip))
w1 = np.sqrt(10.0 / 11.0)
w6 = np.sqrt(1.0 / 11.0)
cboc_e1b = w1 * boc11 - w6 * boc61

tau = np.linspace(-1.2, 1.2, 1200)
r_bpsk1 = np.maximum(0.0, 1.0 - np.abs(tau))
r_boc11 = np.piecewise(np.abs(tau), [np.abs(tau) <= 0.5, (np.abs(tau) > 0.5) & (np.abs(tau) <= 1.0)],
                       [lambda x: 1.0 - 3.0*x, lambda x: x - 1.0, 0.0])
r_bpsk5 = np.maximum(0.0, 1.0 - 5.0 * np.abs(tau))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.2))
ax1.plot(t_chip, cboc_e1b, color='#06b6d4', lw=2, label='E1-B Data CBOC(6,1,1/11, -)')
ax1.plot(t_chip, boc11, color='#94a3b8', ls='--', alpha=0.7, label='Base BOC(1,1)')
ax1.set_title('Galileo E1-B CBOC(6,1,1/11) 4-Level Subcarrier Waveform', fontweight='bold')
ax1.set_xlabel('Time [E1 Chips, 1 chip = 977.5 ns]')
ax1.set_ylabel('Normalized Amplitude')
ax1.legend(loc='upper right')

ax2.plot(tau, r_bpsk1, color='#64748b', ls=':', lw=1.8, label='GPS L1 C/A BPSK(1)')
ax2.plot(tau, r_boc11, color='#06b6d4', lw=2.2, label='Galileo E1 BOC/CBOC')
ax2.plot(tau, r_bpsk5, color='#a855f7', lw=2.2, label='Galileo E6 HAS BPSK(5)')
for tap_x, tap_lbl in [(-0.5, 'VE'), (-0.25, 'E'), (0.0, 'P'), (0.25, 'L'), (0.5, 'VL')]:
    ax2.axvline(tap_x, color='#f59e0b', ls='--', alpha=0.45)
    ax2.text(tap_x, 1.03, tap_lbl, ha='center', fontsize=9, fontweight='bold', color='#d97706')
ax2.set_title('Autocorrelation Function & 5-Tap Correlator (VE, E, P, L, VL)', fontweight='bold')
ax2.set_xlabel('Code Phase Offset τ [E1 Chips]')
ax2.set_ylabel('Normalized Correlation R(τ)')
ax2.set_ylim(-0.6, 1.15)
ax2.legend(loc='upper right')
plt.tight_layout()
plt.show()

---
## 2. Galileo OSNMA v1.1 Cryptographic Engine (Official EUSPA Annex A Vectors)

Per **Galileo OSNMA SIS ICD v1.1** and **Galileo OSNMA Receiver Guidelines v1.3 (Annex A)**, each odd **E1-B I/NAV** half-page carries **40 OSNMA bits** (`8-bit HKROOT` + `32-bit MACK`). Over a 30-second subframe (15 nominal pages) the receiver reconstructs:
- **`HKROOT` Block (120 bits):** Carries `NMAS`, `CID`, `CPKS`, and a fragment of `DSM-PKR` (ECDSA public key authenticated against the **Merkle Tree** root) or `DSM-KROOT` (TESLA root key signed with **ECDSA P-256**).
- **`MACK` Block (480 bits):** Carries truncated `HMAC-SHA-256` tags (`ADKD=0` ephemeris, `ADKD=4` timing/iono, `ADKD=12` Slow MAC), the 12-bit `MACSEQ` sequence integrity field, and the **TESLA** key $K_I$ from the previous subframe.

Below we run the complete **EUSPA Annex A cryptographic verification** (Satellite `E02`, `WN=1248`, `TOW=345660`):

In [ ]:
try:
    from trisat_sdr.nav.osnma_engine import GalileoOsnmaEngine, verify_ecdsa_p256_sha256
    osnma = GalileoOsnmaEngine()
except ImportError:
    osnma = None

ANNEX_A_MERKLE_ROOT = '7332397B4E23D2DFB69580088190B8669D4AC2E8774E0A2B9D3B039784369E68'
ANNEX_A_LEAF_M1     = '040203522563F605C027C3792767E783135AA2D651961C6C19C7B76E912C3258A79D'
ANNEX_A_PDP         = 'B8884C0D33D6'
ANNEX_A_NPK         = '03522563F605C027C3792767E783135AA2D651961C6C19C7B76E912C3258A79D'
ANNEX_A_KROOT_M     = '04221B01610BDF26D77B4E02A35BF8C9CBFCF70422081475FD445DF0FF'
ANNEX_A_KROOT_DS    = (
    '32721996A850C38A8500229D37360D86690B25A132F6F99A51D536D11ADFF72C'
    '8DB9066510DAE21E08F4F5C0B0E7559D77F3798B4D4214C562135E7408D0DB98'
)
ANNEX_A_ALPHA       = '610BDF26D77B'
ANNEX_A_K4          = '69C00AA7364237A65EBF006AD8DDBC73'
ANNEX_A_K0_KROOT    = '5BF8C9CBFCF70422081475FD445DF0FF'

# 1. Verify DSM-PKR Padding P_DP
pdp_calc = hashlib.sha256(bytes.fromhex(ANNEX_A_MERKLE_ROOT + ANNEX_A_LEAF_M1)).digest()[:6].hex().upper()
print(f'[1] DSM-PKR Padding P_DP      : {pdp_calc} == {ANNEX_A_PDP} -> {"✅ OK" if pdp_calc == ANNEX_A_PDP else "❌ FAIL"}')

# 2. Verify TESLA One-Way Chain (K_4 -> K_3 -> K_2 -> K_1 -> K_0 = KROOT)
def tesla_step(k_next_hex: str, wn: int, tow: int, alpha_hex: str) -> str:
    gst32 = ((wn & 0xFFF) << 20) | (tow & 0xFFFFF)
    data = bytes.fromhex(k_next_hex) + gst32.to_bytes(4, 'big') + bytes.fromhex(alpha_hex)
    return hashlib.sha256(data).digest()[:16].hex().upper()

k3 = tesla_step(ANNEX_A_K4, 1248, 345660, ANNEX_A_ALPHA)
k2 = tesla_step(k3,         1248, 345630, ANNEX_A_ALPHA)
k1 = tesla_step(k2,         1248, 345600, ANNEX_A_ALPHA)
k0 = tesla_step(k1,         1248, 345570, ANNEX_A_ALPHA)
print(f'[2] TESLA Chain K4 -> K3      : {k3}')
print(f'                K3 -> K2      : {k2}')
print(f'                K2 -> K1      : {k1}')
print(f'                K1 -> KROOT   : {k0} -> {"✅ OK" if k0 == ANNEX_A_K0_KROOT else "❌ FAIL"}')

# 3. Verify 12-bit MACSEQ at WN=1248, TOW=345660, PRN_A=2
gst_sf = ((1248 & 0xFFF) << 20) | (345660 & 0xFFFFF)
macseq_msg = bytes([2]) + gst_sf.to_bytes(4, 'big') + (0x1205).to_bytes(2, 'big') + (0x050F).to_bytes(2, 'big')
macseq_digest = hmac.new(bytes.fromhex(ANNEX_A_K4), macseq_msg, hashlib.sha256).digest()
macseq_12b = (int.from_bytes(macseq_digest[:2], 'big') >> 4) & 0xFFF
print(f'[3] 12-bit MACSEQ Check       : 0x{macseq_12b:03X} == 0xB08 -> {"✅ OK" if macseq_12b == 0xB08 else "❌ FAIL"}')

# 4. Verify 40-bit HMAC-SHA-256 Tag (ADKD=4)
tag2_msg = bytes.fromhex('02024E05463C03BFFFFFFFC00000449238227897FDEFF93040')
tag2_hex = hmac.new(bytes.fromhex(ANNEX_A_K4), tag2_msg, hashlib.sha256).digest()[:5].hex().upper()
print(f'[4] HMAC-SHA256 Tag (ADKD=4)  : 0x{tag2_hex} == 0x7BB238C883 -> {"✅ OK" if tag2_hex == "7BB238C883" else "❌ FAIL"}')

if osnma is not None:
    ecdsa_ok = verify_ecdsa_p256_sha256(ANNEX_A_NPK, bytes.fromhex(ANNEX_A_KROOT_M), ANNEX_A_KROOT_DS)
    print(f'[5] ECDSA P-256 KROOT Sig     : {"✅ VERIFIED (secp256r1)" if ecdsa_ok else "❌ FAIL"}')

---
## 3. Galileo E6 High Accuracy Service (`EUSPA HAS-UA v2.0`)

The **Galileo HAS Reference User Algorithm Technical Note (`HAS-UA`)** specifies the end-to-end pipeline from **E6-B C/NAV** pages to the **Single-Difference PPP (`PPPSD`)** Extended Kalman Filter:

1. **Reed-Solomon `RS(255, 32, d=224)` Decoding over $\text{GF}(2^8)$:**
   Each C/NAV page delivers one 424-bit HAS word (53 octets in $\text{GF}(2^8)$). Once $k$ distinct pages of the same `MID` are collected, the $k \times k$ Vandermonde submatrix $D_{k \times k}$ is inverted over $\text{GF}(2^8)$:
   $$M = D^{-1} \cdot P$$
2. **Orbit Correction Transformation `NTW` $\to$ `ECEF` (Eq. 19):**
   Using unit vectors $\vec{e}_A = \frac{\vec{v}_{\text{BC}}}{\|\vec{v}_{\text{BC}}\|}$, $\vec{e}_C = \frac{\vec{r}_{\text{BC}} \times \vec{v}_{\text{BC}}}{\|\vec{r}_{\text{BC}} \times \vec{v}_{\text{BC}}\|}$, and $\vec{e}_R = \vec{e}_A \times \vec{e}_C$:
   $$\vec{r}_{\text{HAS}} = \vec{r}_{\text{BC}} - \begin{bmatrix} \vec{e}_R & \vec{e}_A & \vec{e}_C \end{bmatrix} \begin{bmatrix} \Delta r \\ \Delta a \\ \Delta c \end{bmatrix}$$
3. **`DIA` Integrity Control (Detection, Identification, and Adaptation):**
   - **Global `LOM` (Local Overall Model) Test:** $T_{\text{LOM}} = \frac{\hat{v}^T Q_{yy}^{-1} \hat{v}}{m - n} \sim F_{1-\alpha}(m-n, \infty)$.
   - **Baarda Outlier $w$-test:** $|w_i| = \frac{|e_i^T Q_{yy}^{-1} \hat{v}|}{\sqrt{e_i^T Q_{yy}^{-1} Q_{\hat{v}\hat{v}} Q_{yy}^{-1} e_i}} \le N_{1-\alpha_B/2}(0, 1) \approx 3.29$.

In [ ]:
# Simulate EUSPA HAS-UA PPPSD EKF Convergence (SL1 Target: < 20 cm Horizontal / < 40 cm Vertical)
epochs_sec = np.arange(0, 305, 5)
np.random.seed(42)

err_h_os  = 1.35 + 0.18 * np.sin(epochs_sec / 25.0) + np.random.normal(0, 0.06, len(epochs_sec))
err_h_has = 0.13 + 0.85 * np.exp(-epochs_sec / 45.0) + np.random.normal(0, 0.012, len(epochs_sec))
err_v_has = 0.21 + 1.10 * np.exp(-epochs_sec / 50.0) + np.random.normal(0, 0.018, len(epochs_sec))

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(epochs_sec, err_h_os, color='#94a3b8', ls='--', lw=1.8, label='Autonomous Galileo E1 OS (No HAS)')
ax.plot(epochs_sec, err_v_has, color='#a855f7', lw=2.2, label='Galileo E6 HAS-UA PPPSD Vertical Error (1D)')
ax.plot(epochs_sec, err_h_has, color='#10b981', lw=2.6, label='Galileo E6 HAS-UA PPPSD Horizontal Error (2D)')
ax.axhline(0.20, color='#06b6d4', ls=':', lw=1.6, label='EUSPA HAS SL1 Horizontal Target (20 cm, 95%)')
ax.axhline(0.40, color='#f59e0b', ls=':', lw=1.4, label='EUSPA HAS SL1 Vertical Target (40 cm, 95%)')

ax.set_title('EUSPA HAS-UA Convergence (Single-Difference PPPSD Extended Kalman Filter)', fontweight='bold')
ax.set_xlabel('Time from Cold Start [seconds]')
ax.set_ylabel('Positioning Error [meters]')
ax.set_ylim(0, 1.8)
ax.legend(loc='upper right')
plt.tight_layout()
plt.show()

---
## 4. `NTCM-G` 12-Coefficient Ionospheric Model & `EWSS-CAMF` Emergency Alert Geo-Fence

In addition to OSNMA and HAS-UA, **TRISAT** implements:
1. **The `NTCM-G` Ionospheric Model (*Alternate Ionospheric Correction Algorithm for Galileo SF Users*):** Computes Vertical Total Electron Content ($\text{VTEC}$) from 5 multiplicative dependencies ($F_1$ local time/diurnal harmonics, $F_2$ seasonal variation, $F_3$ geomagnetic latitude/equatorial anomaly, $F_4$ geomagnetic crest, and $F_5$ effective solar flux $Az$) and applies the single-layer mapping function at $h_{\text{Iono}} = 450\text{ km}$.
2. **Emergency Warning Satellite Service (`EWSS-CAMF`, 122 bits):** Decodes the WGS84 target area ellipse (fields `A12..A16`: center $(\phi_c, \lambda_c)$, semi-major axis $L_M$, semi-minor axis $L_m$, and azimuth $\theta$) and evaluates whether the receiver is inside the emergency alert zone.

In [ ]:
from trisat_sdr.nav.has_ua_engine import NtcmGIonosphereModel
from trisat_sdr.nav.galileo_advanced_services import GalileoAdvancedServicesEngine

# 1. Evaluate NTCM-G slant ionospheric delay on E1 (1575.42 MHz) vs satellite elevation
elevations = np.linspace(5, 90, 50)
delays_m = [
    NtcmGIonosphereModel.compute_slant_delay_m(40.4168, -3.7038, float(el), 140.0)[0]
    for el in elevations
]

# 2. Visualize the 122-bit EWSS-CAMF emergency alert ellipse around the receiver
adv = GalileoAdvancedServicesEngine()
rep = adv.evaluate(rx_lat=40.4168, rx_lon=-3.7038)
ew = rep.ewss_alert

theta_rad = math.radians(ew.azimuth_deg)
ang = np.linspace(0, 2 * np.pi, 200)
u_km = ew.semi_major_km * np.cos(ang)
v_km = ew.semi_minor_km * np.sin(ang)
dx_km = u_km * np.cos(theta_rad) - v_km * np.sin(theta_rad)
dy_km = u_km * np.sin(theta_rad) + v_km * np.cos(theta_rad)

lat_ellipse = ew.ellipse_center_lat + dy_km / 111.132
lon_ellipse = ew.ellipse_center_lon + dx_km / (111.132 * math.cos(math.radians(ew.ellipse_center_lat)))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.4))
ax1.plot(elevations, delays_m, color='#06b6d4', lw=2.4)
ax1.fill_between(elevations, delays_m, color='#06b6d4', alpha=0.15)
ax1.set_title('NTCM-G Slant Ionospheric Delay on E1 Band (Madrid)', fontweight='bold')
ax1.set_xlabel('Satellite Elevation [degrees]')
ax1.set_ylabel('E1 Ionospheric Slant Delay [meters]')

ax2.plot(lon_ellipse, lat_ellipse, color='#ef4444', lw=2.2, label=f'EWSS Ellipse ({ew.semi_major_km}x{ew.semi_minor_km} km)')
ax2.fill(lon_ellipse, lat_ellipse, color='#ef4444', alpha=0.15)
ax2.scatter([ew.ellipse_center_lon], [ew.ellipse_center_lat], color='#f59e0b', s=70, label='Alert Center (A12/A13)')
ax2.scatter([-3.7038], [40.4168], color='#10b981', s=90, marker='^', label='TRISAT Receiver (Inside: Yes)')
ax2.set_title(f'Galileo EWSS-CAMF Elliptical Geo-Fence ({ew.message_type})', fontweight='bold')
ax2.set_xlabel('WGS84 Longitude [degrees]')
ax2.set_ylabel('WGS84 Latitude [degrees]')
ax2.legend(loc='upper right')
plt.tight_layout()
plt.show()

print('📌 Galileo OS SIS v2.2 Services Summary:')
print(f'   • I/NAV WT16 RedCED : Active (TTFF = {rep.wt16_ttff_sec} s)')
print(f'   • I/NAV WT17-20 RS  : {rep.rs_ced_words_collected}/8 words -> Full CED Reconstructed')
print(f'   • Active SSP Pattern: {rep.current_ssp_id}')
print(f'   • WT22 ARAIM ISM    : {rep.wt22_araim_ism_status}')
print(f'   • WT44 Timing (TSM) : {rep.wt44_tsm_service_level}')
print(f'   • EWSS-CAMF (122b)  : 0x{ew.camf_122bit_hex}')

---
## 5. Full `TrisatSdrReceiver` Epoch Execution & Official 33-Column `.sol` Export

Finally, we execute a complete epoch of the unified `TrisatSdrReceiver` pipeline and output the official **33-column `.sol` solution line** specified in the *Galileo HAS-UA Software User Manual*:

In [ ]:
from trisat_sdr.config import ReceiverConfig
from trisat_sdr.receiver import TrisatSdrReceiver

cfg = ReceiverConfig()
cfg.telemetry.udp_enabled = False
cfg.telemetry.tcp_enabled = False
cfg.telemetry.pty_serial_enabled = False

rx = TrisatSdrReceiver(cfg)
rx.initialize_acquisition()
snapshot = rx.step_epoch()

print('================================================================================')
print(' 🛰️  UNIFIED TRISAT-SDR TELEMETRY SNAPSHOT (OSNMA + HAS-UA + SIS v2.2)')
print('================================================================================')
print(f" PVT Solution Mode : {snapshot['pvt']['solution_mode']}")
print(f" WGS84 Coordinates : ({snapshot['pvt']['latitude_deg']:.6f}, {snapshot['pvt']['longitude_deg']:.6f}) ±{snapshot['pvt']['horizontal_accuracy_m']:.2f} m")
print(f" Pivot Satellites  : Galileo E{snapshot['has_ua']['galileo_pivot_svid']:02d} | GPS G{snapshot['has_ua']['gps_pivot_svid']:02d}")
print(f" DIA Test (LOM/w)  : LOM = {snapshot['has_ua']['lom_test_statistic']} | max|w| = {snapshot['has_ua']['max_baarda_w_test']}")
print(f" OSNMA Annex A     : {'PASSED' if snapshot['osnma']['annex_a_selftest_passed'] else 'FAIL'} (KROOT={snapshot['osnma']['latest_kroot_hex']})")
print('--------------------------------------------------------------------------------')
print(' Official 33-Column EUSPA HAS-UA .sol Line:')
print(snapshot['has_ua']['sol_line_33col'])